# 🛒 Wholesale Customers – Clustering Analysis
**Dataset:** UCI Wholesale Customers (CC0 Public Domain)  
**Goal:** Discover customer segments using K-Means clustering  
**Deliverables:** Elbow curve · Best k · Dunn Index per k · Predicted clusters · Hierarchical clustering (bonus)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cdist
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✓')

---
## 1 · Read the Dataset

In [ ]:
df = pd.read_csv('Wholesale_customers_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

# Drop missing values – one line!
df.dropna(inplace=True)
print(f'\nRows after dropna: {df.shape[0]}')

In [ ]:
df.describe()

---
## 2 · Feature Selection

`Channel` and `Region` are categorical identifiers — using their raw integer values would introduce a false ordinal relationship.  
We **one-hot encode** both so they can meaningfully contribute to the distance calculations used by K-Means.

In [ ]:
df_feat = df.copy()

# One-hot encode Channel and Region
df_feat = pd.get_dummies(df_feat, columns=['Channel', 'Region'], drop_first=False)

print('Feature columns after encoding:')
print(list(df_feat.columns))
df_feat.head()

---
## 3 · Preprocessing

In [ ]:
# StandardScaler: zero mean, unit variance
# Critical so continuous spend columns don't dominate the one-hot dummies
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(df_feat.values)

print(f'Scaled matrix shape: {X_scaled.shape}')
print(f'Mean ≈ {X_scaled.mean():.4f}  |  Std ≈ {X_scaled.std():.4f}')

---
## 4 · K-Means Clustering

### 4a · Dunn Index helper

In [ ]:
def dunn_index(X, labels):
    """
    Dunn Index = min inter-cluster distance / max intra-cluster diameter.
    Higher is better.
    """
    unique_labels = np.unique(labels)
    n_clusters    = len(unique_labels)

    # Intra-cluster diameters (max pairwise distance within each cluster)
    intra = []
    for c in unique_labels:
        members = X[labels == c]
        if len(members) < 2:
            intra.append(0.0)
        else:
            dists = cdist(members, members, metric='euclidean')
            intra.append(dists.max())

    max_intra = max(intra) if max(intra) > 0 else 1e-9

    # Inter-cluster distances (min centroid-to-centroid)
    centroids = np.array([X[labels == c].mean(axis=0) for c in unique_labels])
    inter = []
    for i in range(n_clusters):
        for j in range(i + 1, n_clusters):
            inter.append(np.linalg.norm(centroids[i] - centroids[j]))

    min_inter = min(inter) if inter else 0.0
    return min_inter / max_intra

### 4b · Sweep k, record inertia and Dunn Index

In [ ]:
k_range   = range(2, 11)
inertias  = []
dunn_scores = []

print(f'{"k":<4} {"Inertia":>14}  {"Dunn Index":>12}')
print('-' * 34)

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    iner   = km.inertia_
    dunn   = dunn_index(X_scaled, labels)

    inertias.append(iner)
    dunn_scores.append(dunn)
    print(f'{k:<4} {iner:>14,.2f}  {dunn:>12.4f}')

### 4c · Elbow Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
axes[0].plot(list(k_range), inertias, marker='o', color='steelblue', linewidth=2)
axes[0].set_xlabel('Number of Clusters  k')
axes[0].set_ylabel('Inertia (Within-cluster SS)')
axes[0].set_title('Elbow Curve')
axes[0].set_xticks(list(k_range))

# Dunn Index
axes[1].plot(list(k_range), dunn_scores, marker='s', color='darkorange', linewidth=2)
axes[1].set_xlabel('Number of Clusters  k')
axes[1].set_ylabel('Dunn Index')
axes[1].set_title('Dunn Index vs k  (higher = better)')
axes[1].set_xticks(list(k_range))

plt.suptitle('K-Means Cluster Quality Metrics', fontsize=13)
plt.tight_layout()
plt.show()

### 4d · Choose best k from elbow curve

In [ ]:
# Elbow detection: largest second-derivative (curvature) of the inertia curve
inertia_arr = np.array(inertias)
deltas      = np.diff(inertia_arr)          # first differences
curvature   = np.diff(deltas)               # second differences
best_k      = list(k_range)[np.argmax(curvature) + 1]  # +1 offset for diff

print(f'Best k chosen from elbow curve: k = {best_k}')

### 4e · Final model with best k

In [ ]:
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_best = km_best.fit_predict(X_scaled)

print(f'K-Means with k={best_k}')
print(f'  Inertia    : {km_best.inertia_:,.2f}')
print(f'  Dunn Index : {dunn_index(X_scaled, labels_best):.4f}')
print(f'  Silhouette : {silhouette_score(X_scaled, labels_best):.4f}')
print()
print('Cluster sizes:')
unique, counts = np.unique(labels_best, return_counts=True)
for c, n in zip(unique, counts):
    print(f'  Cluster {c}: {n} customers')

In [ ]:
print('Predicted clusters for the first 10 data samples:')
for i, label in enumerate(labels_best[:10]):
    print(f'  Sample {i:>2} → Cluster {label}')

In [ ]:
# Attach cluster labels and visualise spend profiles
df_result = df.copy()
df_result['Cluster'] = labels_best

spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
cluster_means = df_result.groupby('Cluster')[spend_cols].mean()

cluster_means.T.plot(kind='bar', figsize=(12, 5), colormap='tab10', edgecolor='white')
plt.title(f'Average Spend per Category by Cluster  (k={best_k})', fontsize=13)
plt.xlabel('Product Category')
plt.ylabel('Mean Annual Spend')
plt.legend(title='Cluster')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# 2-D PCA scatter to visualise clusters
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1],
                      c=labels_best, cmap='tab10', s=40, alpha=0.75)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
plt.title(f'K-Means Clusters (k={best_k}) – PCA 2D Projection')
plt.tight_layout()
plt.show()

---
## 5 · Challenge – Hierarchical Clustering

In [ ]:
# Dendrogram (sample 100 points for readability)
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=100, replace=False)
Z = linkage(X_scaled[sample_idx], method='ward')

plt.figure(figsize=(14, 5))
dendrogram(Z, truncate_mode='lastp', p=20,
           leaf_rotation=45, leaf_font_size=9,
           color_threshold=0.7 * max(Z[:, 2]))
plt.title('Hierarchical Clustering Dendrogram (Ward linkage, 100-sample subset)')
plt.xlabel('Sample index (or cluster size)')
plt.ylabel('Ward distance')
plt.tight_layout()
plt.show()

In [ ]:
# Agglomerative Clustering with same best_k
hc = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
hc_labels = hc.fit_predict(X_scaled)

hc_dunn = dunn_index(X_scaled, hc_labels)
hc_sil  = silhouette_score(X_scaled, hc_labels)

print(f'Hierarchical Clustering (Ward, k={best_k})')
print(f'  Dunn Index : {hc_dunn:.4f}')
print(f'  Silhouette : {hc_sil:.4f}')
print()
print('Cluster sizes:')
unique, counts = np.unique(hc_labels, return_counts=True)
for c, n in zip(unique, counts):
    print(f'  Cluster {c}: {n} customers')

In [ ]:
# Side-by-side PCA scatter: K-Means vs Hierarchical
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, lbl, title in zip(axes,
                           [labels_best, hc_labels],
                           [f'K-Means (k={best_k})', f'Hierarchical Ward (k={best_k})']):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                    c=lbl, cmap='tab10', s=35, alpha=0.75)
    plt.colorbar(sc, ax=ax, label='Cluster')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('Clustering Comparison – PCA 2D Projection', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

In [ ]:
print('=' * 50)
print('  CLUSTERING SUMMARY')
print('=' * 50)
print(f'\nDunn Index per k (K-Means):')
for k, d in zip(k_range, dunn_scores):
    marker = ' ← best k' if k == best_k else ''
    print(f'  k={k}  Dunn={d:.4f}{marker}')

print(f'\nBest k from elbow curve : {best_k}')
print(f'\nPredicted clusters (first 10 samples):')
for i, lbl in enumerate(labels_best[:10]):
    print(f'  Sample {i:>2} → Cluster {lbl}')

print(f'\nModel comparison at k={best_k}:')
print(f'  K-Means      Dunn={dunn_index(X_scaled, labels_best):.4f}  Silhouette={silhouette_score(X_scaled, labels_best):.4f}')
print(f'  Hierarchical Dunn={hc_dunn:.4f}  Silhouette={hc_sil:.4f}')
print('=' * 50)